# 04 Limpieza 07 — Normalizar nombres de autores

Esta libreta aplica el diccionario final corregido de aliases de autores a la columna `Autor_norm`.

- Entrada: `integrado_solo_unam_estandarizado_pre_deduplicacion_limpio.csv`
- Diccionario: `diccionario_alias_autores_unam_final_corregido.csv`
- Salida única: `integrado_solo_unam_autores_normalizados.csv`

La libreta mantiene las 14 columnas canónicas, no deduplica, no fusiona registros y no modifica afiliaciones ni áreas. También aplica la exclusión manual indicada para `R Ramos` en el DOI `10.1109/oceans55160.2024.10754153`.

In [1]:
"""
Fase 03_Nomralizar_Nombres: normalizar Autor_norm con el diccionario final corregido.

Entrada principal:
- C:\\Users\\hazar\\Desktop\\PROYECTO\\04_Limpieza\\02_Normalizar_Columnas\\integrado_solo_unam_estandarizado_pre_deduplicacion_limpio.csv

Diccionario:
- C:\\Users\\hazar\\Desktop\\PROYECTO\\04_Limpieza\\03_Nomralizar_Nombres\\diccionario_alias_autores_unam_final_corregido.csv

Salida única:
- C:\\Users\\hazar\\Desktop\\PROYECTO\\04_Limpieza\\03_Nomralizar_Nombres\\integrado_solo_unam_autores_normalizados.csv

Criterios:
- Mantiene exactamente las 14 columnas canónicas.
- Sólo reemplaza Autor_norm usando el diccionario alias -> autor_normalizado.
- No modifica Afiliacion1, Afiliacion2, Area, Subarea, DOI, URL, ISBN, ISSN, Titulo, Keywords ni Abstract.
- No deduplica ni fusiona.
- Aplica la exclusión manual indicada: R Ramos del DOI 10.1109/oceans55160.2024.10754153 no es UNAM y se elimina.
"""

from pathlib import Path
from collections import defaultdict, Counter
import os
import re
import unicodedata
import pandas as pd

In [2]:
# ---------------------------------------------------------------------
# 1. Rutas
# ---------------------------------------------------------------------
RAIZ_PROYECTO = Path(r"C:\Users\hazar\Desktop\PROYECTO")

ARCHIVO_ENTRADA = RAIZ_PROYECTO / r"04_Limpieza\02_Normalizar_Columnas\integrado_solo_unam_estandarizado_pre_deduplicacion_limpio.csv"
ARCHIVO_DICCIONARIO = RAIZ_PROYECTO / r"04_Limpieza\03_Nomralizar_Nombres\diccionario_alias_autores_unam_final_corregido.csv"
CARPETA_SALIDA = RAIZ_PROYECTO / r"04_Limpieza\03_Nomralizar_Nombres"
ARCHIVO_SALIDA = CARPETA_SALIDA / "integrado_solo_unam_autores_normalizados.csv"


if os.name != "nt" and not ARCHIVO_ENTRADA.exists():
    ARCHIVO_ENTRADA = Path("/mnt/data/integrado_solo_unam_estandarizado_pre_deduplicacion_limpio.csv")
if os.name != "nt" and not ARCHIVO_DICCIONARIO.exists():
    ARCHIVO_DICCIONARIO = Path("/mnt/data/diccionario_alias_autores_unam_final_corregido.csv")
if os.name != "nt" and not CARPETA_SALIDA.exists():
    CARPETA_SALIDA = Path("/mnt/data/03_Nomralizar_Nombres")
    ARCHIVO_SALIDA = CARPETA_SALIDA / "integrado_solo_unam_autores_normalizados.csv"

CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

COLUMNAS_CANONICAS = [
    "indice", "Titulo", "Año", "Autor_norm", "Afiliacion1", "Afiliacion2",
    "ISBN", "ISSN", "Doi", "URL", "Area", "Subarea", "Keywords", "Abstract",
]

In [3]:
# ---------------------------------------------------------------------
# 2. Funciones auxiliares
# ---------------------------------------------------------------------
def limpiar_espacios(valor: object) -> str:
    texto = "" if valor is None else str(valor)
    texto = texto.replace("\ufeff", " ").replace("\r", " ").replace("\n", " ").replace("\t", " ")
    return re.sub(r"\s+", " ", texto).strip()


def quitar_acentos(texto: object) -> str:
    texto = "" if texto is None else str(texto)
    return "".join(
        c for c in unicodedata.normalize("NFKD", texto)
        if not unicodedata.combining(c)
    )


def clave_autor(texto: object) -> str:
    """
    Clave interna para comparar aliases de autores.
    No se escribe en la base final.
    """
    texto = limpiar_espacios(texto)
    texto = quitar_acentos(texto).lower()
    texto = re.sub(r"https?://\S+|\S+@\S+", " ", texto)
    texto = re.sub(r"\bORCID\b.*$", " ", texto, flags=re.I)
    texto = re.sub(r"\[[^\]]*\]|\([^)]*\)", " ", texto)
    texto = texto.replace("‐", "-").replace("‑", "-").replace("–", "-").replace("—", "-")
    texto = texto.replace("-", " ")
    texto = re.sub(r"[.,;:!¡?¿\"“”‘’`´*]", " ", texto)
    texto = re.sub(r"\d+", " ", texto)
    texto = re.sub(r"[^a-zñáéíóúü0-9]+", " ", texto)
    return re.sub(r"\s+", " ", texto).strip()


def autor_a_formato_nombre_apellido(texto: object) -> str:
    """
    Convierte internamente 'Apellido, Nombre' a 'Nombre Apellido' para buscar variantes.
    No modifica directamente la base final; sólo ayuda a buscar en el diccionario.
    """
    texto = limpiar_espacios(texto)
    if "," in texto:
        apellido, nombres = texto.split(",", 1)
        return limpiar_espacios(f"{nombres.strip()} {apellido.strip()}")
    return texto


def doi_limpio(valor: object) -> str:
    texto = limpiar_espacios(valor).lower()
    texto = re.sub(r"^https?://(?:dx\.)?doi\.org/", "", texto)
    texto = re.sub(r"^doi\s*[:/]\s*", "", texto)
    m = re.search(r"10\.\d{4,9}/[^\s,;<>\"]+", texto)
    return m.group(0).strip(" .;,)]}") if m else texto.strip(" .;,)]}")


def validar_columnas(df: pd.DataFrame, nombre: str) -> None:
    faltantes = [c for c in COLUMNAS_CANONICAS if c not in df.columns]
    extras = [c for c in df.columns if c not in COLUMNAS_CANONICAS]
    if faltantes or extras:
        raise ValueError(
            f"{nombre} no tiene exactamente las 14 columnas canónicas. "
            f"Faltantes={faltantes}; Extras={extras}"
        )

In [4]:
# ---------------------------------------------------------------------
# 3. Carga
# ---------------------------------------------------------------------
df = pd.read_csv(ARCHIVO_ENTRADA, dtype=str, keep_default_na=False, encoding="utf-8-sig")
dic = pd.read_csv(ARCHIVO_DICCIONARIO, dtype=str, keep_default_na=False, encoding="utf-8-sig")

validar_columnas(df, "La base de entrada")

columnas_dic_requeridas = {"alias", "autor_normalizado"}
if not columnas_dic_requeridas.issubset(dic.columns):
    raise ValueError(
        f"El diccionario debe tener al menos las columnas {columnas_dic_requeridas}. "
        f"Columnas recibidas: {list(dic.columns)}"
    )

df = df[COLUMNAS_CANONICAS].copy()
for col in COLUMNAS_CANONICAS:
    df[col] = df[col].map(limpiar_espacios)

dic["alias"] = dic["alias"].map(limpiar_espacios)
dic["autor_normalizado"] = dic["autor_normalizado"].map(limpiar_espacios)
dic = dic[(dic["alias"] != "") & (dic["autor_normalizado"] != "")].copy()

In [5]:
# ---------------------------------------------------------------------
# 4. Construir mapa flexible alias -> autor_normalizado
# ---------------------------------------------------------------------
claves_a_autores = defaultdict(set)

for _, fila in dic.iterrows():
    alias = fila["alias"]
    autor_normalizado = fila["autor_normalizado"]

    posibles_alias = [
        alias,
        autor_a_formato_nombre_apellido(alias),
        autor_normalizado,
    ]

    for posible in posibles_alias:
        k = clave_autor(posible)
        if k:
            claves_a_autores[k].add(autor_normalizado)

# Separar claves seguras y ambiguas.
claves_ambiguas = {
    clave: sorted(valores)
    for clave, valores in claves_a_autores.items()
    if len(valores) > 1
}

mapa_seguro = {
    clave: next(iter(valores))
    for clave, valores in claves_a_autores.items()
    if len(valores) == 1
}

if claves_ambiguas:
    print("Advertencia: se detectaron claves ambiguas en el diccionario. No se usarán para normalizar.")
    print(f"Claves ambiguas: {len(claves_ambiguas):,}")
    for i, (clave, valores) in enumerate(list(claves_ambiguas.items())[:10], start=1):
        print(f"{i}. {clave} -> {valores}")

Advertencia: se detectaron claves ambiguas en el diccionario. No se usarán para normalizar.
Claves ambiguas: 1
1. armando castaneda -> ['Armando Castañeda', 'Armando Castañeda Rojano']


In [6]:
# ---------------------------------------------------------------------
# 5. Aplicar normalización de Autor_norm
# ---------------------------------------------------------------------
autores_originales = df["Autor_norm"].copy()
normalizados = []
sin_cobertura = []

for idx, autor in autores_originales.items():
    candidatos_clave = [
        clave_autor(autor),
        clave_autor(autor_a_formato_nombre_apellido(autor)),
    ]

    autor_final = ""
    for k in candidatos_clave:
        if k in mapa_seguro:
            autor_final = mapa_seguro[k]
            break

    if not autor_final:
        sin_cobertura.append({
            "fila": idx,
            "indice": df.at[idx, "indice"],
            "Autor_norm_original": autor,
            "Titulo": df.at[idx, "Titulo"],
            "Doi": df.at[idx, "Doi"],
            "Afiliacion1": df.at[idx, "Afiliacion1"],
            "Afiliacion2": df.at[idx, "Afiliacion2"],
        })
        autor_final = autor  # No se inventa. Se conserva si no hay cobertura.

    normalizados.append(autor_final)

df["Autor_norm"] = normalizados

if sin_cobertura:
    print("Autores sin cobertura en el diccionario corregido:")
    for item in sin_cobertura[:20]:
        print(item)
    raise ValueError(
        f"Hay {len(sin_cobertura):,} filas sin cobertura en el diccionario de autores. "
        "Corrige el diccionario antes de generar la base final."
    )

In [7]:
# ---------------------------------------------------------------------
# 6. Exclusiones manuales justificadas antes de guardar
# ---------------------------------------------------------------------
# Caso indicado manualmente:
# R Ramos en el artículo con DOI 10.1109/oceans55160.2024.10754153 no es autor afiliado a UNAM.
EXCLUSIONES_MANUALES = [
    {
        "doi": "10.1109/oceans55160.2024.10754153",
        "alias_originales": ["Ramos, R.", "R Ramos"],
        "motivo": "R Ramos no es afiliado UNAM en este artículo específico.",
    }
]

mascara_eliminar = pd.Series(False, index=df.index)

for regla in EXCLUSIONES_MANUALES:
    doi_objetivo = doi_limpio(regla["doi"])
    alias_keys = {clave_autor(a) for a in regla["alias_originales"]}

    mascara_regla = (
        df["Doi"].map(doi_limpio).eq(doi_objetivo)
        & autores_originales.map(clave_autor).isin(alias_keys)
    )

    n_eliminar = int(mascara_regla.sum())
    if n_eliminar:
        print(f"Exclusión manual aplicada: {regla['motivo']} | filas eliminadas: {n_eliminar}")
    else:
        print(f"Exclusión manual sin coincidencias: {regla['motivo']}")

    mascara_eliminar |= mascara_regla

filas_antes_exclusion = len(df)
df = df.loc[~mascara_eliminar].copy()
filas_eliminadas = filas_antes_exclusion - len(df)

Exclusión manual aplicada: R Ramos no es afiliado UNAM en este artículo específico. | filas eliminadas: 2


In [8]:
# ---------------------------------------------------------------------
# 7. Validaciones finales
# ---------------------------------------------------------------------
# Mantener sólo columnas canónicas y en orden.
df = df[COLUMNAS_CANONICAS].copy()

# Subarea debe seguir vacía.
df["Subarea"] = ""

# Autor_norm ya no debe contener comas, puntos, guiones, números ni marcadores.
patron_autor_sospechoso = re.compile(r"[,.;:\[\]\(\)\*\d]")
autores_sospechosos = df[df["Autor_norm"].map(lambda x: bool(patron_autor_sospechoso.search(x or "")))]

if not autores_sospechosos.empty:
    print("Autores normalizados con formato sospechoso:")
    print(autores_sospechosos[["indice", "Autor_norm", "Titulo", "Doi"]].head(20).to_string(index=False))
    raise ValueError(
        f"Hay {len(autores_sospechosos):,} filas con Autor_norm sospechoso. "
        "Revisa el diccionario antes de guardar."
    )

if list(df.columns) != COLUMNAS_CANONICAS:
    raise ValueError("La salida no conserva exactamente las 14 columnas canónicas.")

In [9]:
# ---------------------------------------------------------------------
# 8. Guardar salida única
# ---------------------------------------------------------------------
df.to_csv(ARCHIVO_SALIDA, index=False, encoding="utf-8-sig")

In [10]:
# ---------------------------------------------------------------------
# 9. Resumen de control en pantalla
# ---------------------------------------------------------------------
print("Normalización de autores terminada.")
print(f"Archivo de entrada: {ARCHIVO_ENTRADA}")
print(f"Diccionario usado: {ARCHIVO_DICCIONARIO}")
print(f"Archivo de salida: {ARCHIVO_SALIDA}")
print()
print(f"Filas originales: {len(autores_originales):,}")
print(f"Filas eliminadas por exclusión manual: {filas_eliminadas:,}")
print(f"Filas finales: {len(df):,}")
print(f"Autores únicos originales: {autores_originales.nunique():,}")
print(f"Autores únicos finales: {df['Autor_norm'].nunique():,}")
print(f"Columnas finales: {len(df.columns)}")
print(f"Subarea vacía: {(df['Subarea'].str.strip() == '').all()}")
print(f"Autores con coma/punto/número/marcador: {len(autores_sospechosos):,}")

Normalización de autores terminada.
Archivo de entrada: C:\Users\hazar\Desktop\PROYECTO\04_Limpieza\02_Normalizar_Columnas\integrado_solo_unam_estandarizado_pre_deduplicacion_limpio.csv
Diccionario usado: C:\Users\hazar\Desktop\PROYECTO\04_Limpieza\03_Nomralizar_Nombres\diccionario_alias_autores_unam_final_corregido.csv
Archivo de salida: C:\Users\hazar\Desktop\PROYECTO\04_Limpieza\03_Nomralizar_Nombres\integrado_solo_unam_autores_normalizados.csv

Filas originales: 3,239
Filas eliminadas por exclusión manual: 2
Filas finales: 3,237
Autores únicos originales: 1,112
Autores únicos finales: 743
Columnas finales: 14
Subarea vacía: True
Autores con coma/punto/número/marcador: 0
